# Build Your Own AI Study Assistant with Google ADK
### Build With AI — Google Developer Groups

---

In this codelab you will build **StudyBot** — an AI study assistant that can:
- Explain any topic at your level
- Quiz you with multiple choice questions
- Build a day-by-day study plan for your exam

You will use **Google ADK** (Agent Development Kit) and **Gemini 2.0 Flash**.

By the end you will have a working AI agent running in your browser.

---

### What is Agentic AI?

A normal chatbot only responds to messages.

An AI agent can respond and also take actions — it can search the web, run code, call APIs, and decide on its own what steps to take to solve a problem.

| Normal Chatbot | AI Agent |
|---|---|
| Responds to messages | Responds and takes actions |
| No tools | Has tools it can call |
| Single step | Multi-step reasoning |

### What is Google ADK?

Google ADK (Agent Development Kit) is Google's open-source Python framework for building AI agents. It handles the agent loop, tool calling, session memory and deployment so you can focus on building the agent rather than the infrastructure around it.

---

### Before you start

You need:
- A Google account
- A GitHub account
- This repository open in GitHub Codespaces
- A Gemini API key (covered in Step 2)

**What is GitHub Codespaces?**

A coding environment that runs entirely in your browser. No installations needed on your laptop — everything runs in the cloud.

---
## Step 1 — Set Up Your Environment

Open the Terminal in your Codespace (bottom panel) and run the cell below.

**How to use this notebook:**
- Code cells have `[ ]` on the left — click one and press Shift + Enter to run it
- Or copy the command and paste it directly into the terminal
- Read each section fully before running the code

In [ ]:
# Install Google ADK
# This takes about a minute — wait for it to finish before moving on
!pip install google-adk

In [ ]:
# Verify the installation
# Expected output: adk, version 1.x.x
!adk --version

### Common errors at this step

**`command not found: adk`**

ADK did not install properly. Run `pip install google-adk` again and wait for it to complete fully.

**`pip: command not found`**

Try using pip3 instead: `pip3 install google-adk`

**`Permission denied`**

Add `--user` to the command: `pip install google-adk --user`

---
## Step 2 — Get Your Gemini API Key

Your agent needs a key to connect to Gemini. Getting one is free.

1. Go to https://aistudio.google.com/apikey
2. Sign in with your Google account
3. Click **Create API Key**
4. Copy the key — it starts with `AIzaSy...`

**Keep your API key private.**

Do not share it publicly, do not paste it in a chat, do not push it to GitHub. Treat it the same as a password.

Now create the project folder and files:

In [ ]:
# Create the project structure
!mkdir -p studybot
!touch studybot/agent.py
!touch studybot/__init__.py
!touch studybot/tools.py
!touch studybot/.env
!touch .gitignore

print("Files created.")

Open `studybot/.env` in your editor and add these two lines:

```
GOOGLE_API_KEY=paste_your_key_here
GOOGLE_GENAI_USE_VERTEXAI=FALSE
```

Replace `paste_your_key_here` with your actual key from AI Studio.

**What is a `.env` file?**

It stores secret values like API keys. It stays on your machine only and should never be committed to GitHub.

Open `.gitignore` and add these lines:

```
.env
venv/
__pycache__/
*.pyc
```

### Common errors at this step

**`GOOGLE_GENAI_USE_VERTEXAI=FALS` — missing the E**

It must be exactly `FALSE`. One missing character will break authentication.

**Accidentally shared your API key?**

Go to https://aistudio.google.com/apikey, delete that key immediately and create a new one.

---
## Step 3 — Write Your First ADK Agent

Open `studybot/agent.py` and paste in the code from the cell below.

**What is `root_agent`?**

ADK looks for a variable named exactly `root_agent` in your `agent.py` file. Do not rename it.

**What is `instruction`?**

This is the system prompt. It tells the AI who it is and how to behave. This is the main thing you will customize later.

**What does `_load_local_env()` do?**

It reads your `.env` file and loads the API key into the environment. ADK in Codespaces does not always pick this up automatically, so this function handles it manually.

In [ ]:
# Paste everything printed below into studybot/agent.py
# Do not run this as a notebook cell — open the file and paste it there

code = '''
import os
from pathlib import Path
from google.adk.agents import Agent


def _load_local_env() -> None:
    """Load variables from studybot/.env if they are not already set."""
    env_path = Path(__file__).with_name(".env")
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip(\'"\').strip("\'")
        if key:
            os.environ.setdefault(key, value)


_load_local_env()

root_agent = Agent(
    model="gemini-2.0-flash",
    name="StudyBot",
    description="A helpful AI study assistant for students.",
    instruction="""You are StudyBot, a friendly study assistant.
    You help students understand topics clearly, create study plans,
    and quiz them to test their knowledge.
    Always be encouraging and break down complex topics into simple steps.""",
)
'''

print(code)

### Run your agent

In the terminal, run this from the root of your project — not from inside the studybot folder:

```bash
adk web . --host 127.0.0.1 --port 8000 --allow_origins "regex:.*"
```

You should see:
```
INFO: Application startup complete.
INFO: Uvicorn running on http://127.0.0.1:8000
```

Open the URL in your browser. Then:
1. Turn **Token Streaming off** using the toggle in the top right corner
2. Click **+ New Session**
3. Send this message: `Explain recursion to me as a beginner`

### Common errors at this step

**`403 Forbidden` when sending a message**

The `--allow_origins "regex:.*"` part is missing from the run command. Stop the server with Ctrl + C and restart with the full command above.

**`address already in use` on port 8000**

Something else is using port 8000. Free it with:
```bash
kill -9 $(lsof -t -i:8000)
```
Then restart the server.

**Messages send but nothing comes back**

Token Streaming is on. Turn it off in the top right corner, start a new session and try again.

**`root_agent not found`**

The variable must be named exactly `root_agent` — not `agent`, not `my_agent`.

**Port not accessible in GitHub Codespaces**

Go to the Ports tab at the bottom of VS Code. Find port 8000, right-click and set Port Visibility to Public.

---
## Step 4 — Add Your First Tool: explain_topic

StudyBot can chat but it cannot take actions yet. Tools are what turn a chatbot into an agent.

A tool in ADK is a regular Python function. ADK and Gemini figure out when to call it by reading the **docstring** — the text inside the triple quotes at the top of the function. Write it clearly because Gemini reads it to understand what the tool does and what inputs it expects.

Open `studybot/tools.py` and paste this in:

In [ ]:
# Paste this into studybot/tools.py

code = '''
def explain_topic(topic: str, level: str = "beginner") -> str:
    """Explains a study topic at a given difficulty level.

    Args:
        topic: The subject or concept the student wants to understand.
        level: The difficulty level - either beginner, intermediate, or advanced.

    Returns:
        A structured explanation of the topic at the requested level.
    """
    return f"Please explain {topic} at a {level} level with examples and key points."
'''

print(code)

Now open `studybot/agent.py` and make two changes.

Add this import after the existing imports:
```python
from studybot.tools import explain_topic
```

Add `tools=[explain_topic]` inside the Agent:
```python
root_agent = Agent(
    model="gemini-2.0-flash",
    name="StudyBot",
    description="A helpful AI study assistant for students.",
    instruction="""You are StudyBot, a friendly study assistant.
    When a student asks you to explain something, use the explain_topic tool.
    Always be encouraging and clear.""",
    tools=[explain_topic],
)
```

Stop the server, restart it, then send:
`Explain machine learning to me at a beginner level`

Look at the **Trace panel** on the left side of the ADK UI. You should see `explain_topic` listed there — that confirms the agent called the tool.

### Common errors at this step

**`ImportError: cannot import name explain_topic`**

The function name in `tools.py` does not match what you are importing. Check the spelling — Python is case-sensitive.

**Tool not showing in the Trace panel**

Either `tools=[explain_topic]` is missing from the Agent, or the server was not restarted. Changes only take effect after a full restart.

**`ModuleNotFoundError: No module named studybot`**

The file `studybot/__init__.py` is missing. Create it — it can be empty, it just needs to exist.

---
## Step 5 — Add the Quiz Tool

This tool generates multiple choice questions on any topic and marks the student's answers.

Open `studybot/tools.py` and add this function below `explain_topic`:

In [ ]:
# Add this to studybot/tools.py — place it below explain_topic

code = '''
def quiz_student(topic: str) -> str:
    """Generates a short quiz to test a student's knowledge on a topic.

    Args:
        topic: The subject the student wants to be quizzed on.

    Returns:
        A set of 3 multiple choice questions with options A, B, C, D.
    """
    return f"""Generate exactly 3 multiple choice questions about {topic}.
    Format each question like this:

    Q1: [question]
    A) [option]
    B) [option]
    C) [option]
    D) [option]
    Answer: [correct letter]

    After the student answers, mark them and give feedback."""
'''

print(code)

Update the import in `agent.py`:
```python
from studybot.tools import explain_topic, quiz_student
```

Update the tools list:
```python
tools=[explain_topic, quiz_student],
```

Restart the server and test with:
`Quiz me on Python functions`

### Common errors at this step

**Agent still behaving the same after the change**

Stop the server fully with Ctrl + C before restarting. A running server does not pick up code changes.

**Port still busy after stopping**

Run `kill -9 $(lsof -t -i:8000)` to force-release the port, then restart.

---
## Step 6 — Add the Study Planner Tool

The student gives it a subject and the number of days until their exam. The agent returns a full day-by-day study schedule.

Add this to `studybot/tools.py` below the quiz tool:

In [ ]:
# Add this to studybot/tools.py — place it below quiz_student

code = '''
def study_planner(subject: str, days_until_exam: int) -> str:
    """Creates a day-by-day study plan for a student preparing for an exam.

    Args:
        subject: The subject the student is preparing for.
        days_until_exam: The number of days the student has until their exam.

    Returns:
        A structured day-by-day study schedule with topics and goals.
    """
    return f"""Create a detailed {days_until_exam}-day study plan for {subject}.

    Format it like this:

    Day 1: [Topic to cover]
    - Goal: [What the student should achieve]
    - Resources: [What to read or watch]
    - Practice: [Exercise to do]

    Repeat for each day. End with exam-day tips."""
'''

print(code)

Update the import in `agent.py`:
```python
from studybot.tools import explain_topic, quiz_student, study_planner
```

Update the tools list:
```python
tools=[explain_topic, quiz_student, study_planner],
```

Restart and test with:
`Create a 5 day study plan for Python`

All three tools should now appear in the Trace panel.

---
## Step 7 — The Complete agent.py

Here is the complete final version of `agent.py` with all three tools wired in. Make sure yours matches this before moving on.

In [ ]:
# Complete final agent.py
# Paste this into studybot/agent.py

code = '''
import os
from pathlib import Path
from google.adk.agents import Agent
from studybot.tools import explain_topic, quiz_student, study_planner


def _load_local_env() -> None:
    """Load variables from studybot/.env if they are not already set."""
    env_path = Path(__file__).with_name(".env")
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip(\'"\').strip("\'")
        if key:
            os.environ.setdefault(key, value)


_load_local_env()

root_agent = Agent(
    model="gemini-2.0-flash",
    name="StudyBot",
    description="A helpful AI study assistant for students.",
    instruction="""You are StudyBot, a friendly study assistant.
    When a student asks you to explain something, use the explain_topic tool.
    When a student wants to be tested, use the quiz_student tool.
    When a student needs a study plan, use the study_planner tool.
    Always be clear and supportive.""",
    tools=[explain_topic, quiz_student, study_planner],
)
'''

print(code)

---
## Step 8 — Customize Your Agent

The `instruction` string controls the agent's personality and focus. Swap it out and you have a completely different assistant.

Try one of these:

### Math tutor
```python
instruction="""You are MathBot, a patient math tutor.
You help students understand mathematics from basic arithmetic to calculus.
Always show step-by-step working and use simple examples."""
```

### Coding mentor
```python
instruction="""You are CodeMentor, a programming tutor.
You help students learn Python, JavaScript and web development.
Always explain with working code examples."""
```

### Swahili study assistant
```python
instruction="""Wewe ni StudyBot, msaidizi wa masomo wa AI.
Jibu maswali yote kwa Kiswahili.
Eleza mambo kwa urahisi na toa mifano inayoeleweka."""
```

### Strict exam mode
```python
instruction="""You are ExamBot, a strict exam coach.
You only quiz students — no hints, no giving answers away.
Only reveal the correct answer after the student has attempted."""
```

Write your own for a subject you care about. Change the name, the personality and the focus. Restart the server and test it.

---
## Step 9 — Push to GitHub

Save your work to GitHub so you can share it and return to it later.

Before pushing, confirm `.env` is in your `.gitignore`:

In [ ]:
# Confirm .env is protected before pushing
!cat .gitignore

# If .env is not listed, run this:
# echo ".env" >> .gitignore

In [ ]:
!git status

In [ ]:
!git add .

In [ ]:
!git commit -m "feat: add StudyBot ADK agent with explain, quiz and study planner tools"

In [ ]:
!git push origin main

### Common errors at this step

**`Authentication failed`**

In Codespaces this usually resolves on its own. If not, go to GitHub Settings, then Developer Settings, then Personal Access Tokens and create a new token.

**`nothing to commit`**

Your files have not been saved. Press Ctrl + S on each open file in VS Code before running git add.

---
## Step 10 — Where to Take This Next

You have a working agentic AI application. Here are some directions to explore.

### Add Google Search

ADK includes a built-in Google Search tool. Add it with two lines:

```python
from google.adk.tools import google_search

tools=[explain_topic, quiz_student, study_planner, google_search],
```

Your agent can now search the web when answering questions.

### Deploy to Google Cloud Run

Get a live public HTTPS URL:

```bash
gcloud run deploy studybot --source . --region us-central1
```

### Build a multi-agent system

Create separate specialist agents — a QuizAgent, a PlannerAgent, a SearchAgent — and have a root agent coordinate them. ADK supports this without any extra libraries.

### Use the Eval tab

The ADK web UI has a built-in Eval tab. Write test cases and measure how consistently your agent performs across different inputs.

---

### Links

- Google ADK Documentation: https://google.github.io/adk-docs/
- Google AI Studio API Keys: https://aistudio.google.com/apikey
- Google Cloud Console: https://console.cloud.google.com
- This repo: https://github.com/Obila34/pyforge-labs

---

## Done

You built a working AI agent from scratch using Google ADK and Gemini.

You now understand:
- What agentic AI is and how it differs from a regular chatbot
- How Google ADK works and why it simplifies agent development
- How to write tools and wire them into an agent
- How to customize an agent's persona through the instruction prompt
- How to save and share your work using GitHub

Push your repo and share what you built.